# Session 5 - LangGraph Fundamentals

AI Agents & Workflows: Basics

In this notebook we will build the fundamentals of LangGraph:

- simple graphs
- state
- nodes
- edges
- conditional routing
- message-based chatbot graph
- short-term memory with checkpointer
- practical workflow example

Main mental model:

```text
State = what we know
Node = what we do
Edge = where we go next
```

## 0. Setup

This notebook expects the same project setup as previous sessions:

- `.env` file in the project root
- `OPENAI_API_KEY` inside `.env`
- optional `OPENAI_MODEL` inside `.env`

Example:

```bash
OPENAI_API_KEY=your_api_key_here
OPENAI_MODEL=gpt-4o-mini
```

Required packages:

```bash
pip install langgraph langchain langchain-openai python-dotenv
```

In [ ]:
import os
from dotenv import load_dotenv

# The notebook is inside notebooks/, so we load ../.env from the project root.
load_dotenv("../.env")

api_key = os.getenv("OPENAI_API_KEY")
model_name = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

print("OPENAI_API_KEY loaded:", bool(api_key))
print("Model:", model_name)

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

print("LangGraph imports OK")

## 1. Why LangGraph?

LangChain is great for:

- model calls
- prompt templates
- tools
- simple chains
- simple agents

LangGraph is useful when we need:

- state
- multiple steps
- branching
- loops
- memory
- human approval
- controlled workflow execution

A graph gives us explicit control.

## 2. Simplest Possible Graph

A graph has:

```text
START → node → END
```

The node is just a Python function.

It receives state and returns state updates.

In [ ]:
class SimpleState(TypedDict):
    message: str
    result: str


def uppercase_node(state: SimpleState):
    """A simple node that reads state and returns an update."""
    message = state["message"]
    return {"result": message.upper()}


builder = StateGraph(SimpleState)

builder.add_node("uppercase", uppercase_node)

builder.add_edge(START, "uppercase")
builder.add_edge("uppercase", END)

simple_graph = builder.compile()

print("Simple graph compiled")

In [ ]:
output = simple_graph.invoke({
    "message": "hello langgraph",
    "result": ""
})

output

### What happened?

Input state:

```python
{"message": "hello langgraph", "result": ""}
```

The node returned:

```python
{"result": "HELLO LANGGRAPH"}
```

LangGraph merged this update into the final state.

## 3. Multi-Step Graph

Now let's create a graph with multiple nodes:

```text
START → prepare → process → finish → END
```

Each node updates the same state.

In [ ]:
class WorkflowState(TypedDict):
    text: str
    prepared_text: str
    processed_text: str
    final_output: str


def prepare_node(state: WorkflowState):
    return {"prepared_text": state["text"].strip()}


def process_node(state: WorkflowState):
    return {"processed_text": state["prepared_text"].upper()}


def finish_node(state: WorkflowState):
    return {"final_output": f"Final result: {state['processed_text']}"}


builder = StateGraph(WorkflowState)

builder.add_node("prepare", prepare_node)
builder.add_node("process", process_node)
builder.add_node("finish", finish_node)

builder.add_edge(START, "prepare")
builder.add_edge("prepare", "process")
builder.add_edge("process", "finish")
builder.add_edge("finish", END)

workflow_graph = builder.compile()

print("Workflow graph compiled")

In [ ]:
workflow_graph.invoke({
    "text": "   langgraph workflow   ",
    "prepared_text": "",
    "processed_text": "",
    "final_output": ""
})

## 4. Stream State Updates

Instead of only receiving the final state, we can stream updates.

This is useful for debugging and observability.

In [ ]:
for update in workflow_graph.stream(
    {
        "text": "   stream this workflow   ",
        "prepared_text": "",
        "processed_text": "",
        "final_output": ""
    },
    stream_mode="updates"
):
    print(update)

## 5. Conditional Routing

Conditional edges allow the graph to choose the next node based on state.

Example:

```text
classify_ticket
    ↓
incident_handler OR docs_handler OR general_handler
```

This is one of the most important LangGraph patterns.

In [ ]:
class TicketState(TypedDict):
    ticket: str
    category: str
    response: str


def classify_ticket(state: TicketState):
    ticket = state["ticket"].lower()

    if any(word in ticket for word in ["down", "failed", "error", "incident", "production"]):
        category = "incident"
    elif any(word in ticket for word in ["documentation", "readme", "guide", "docs"]):
        category = "documentation"
    else:
        category = "general"

    return {"category": category}


def incident_handler(state: TicketState):
    return {"response": "Incident detected. Collect logs, check monitoring, and escalate if needed."}


def documentation_handler(state: TicketState):
    return {"response": "Documentation request detected. Search or update the relevant guide."}


def general_handler(state: TicketState):
    return {"response": "General request detected. Ask for more context if needed."}


def route_by_category(state: TicketState) -> Literal["incident", "documentation", "general"]:
    return state["category"]


builder = StateGraph(TicketState)

builder.add_node("classify_ticket", classify_ticket)
builder.add_node("incident_handler", incident_handler)
builder.add_node("documentation_handler", documentation_handler)
builder.add_node("general_handler", general_handler)

builder.add_edge(START, "classify_ticket")

builder.add_conditional_edges(
    "classify_ticket",
    route_by_category,
    {
        "incident": "incident_handler",
        "documentation": "documentation_handler",
        "general": "general_handler",
    }
)

builder.add_edge("incident_handler", END)
builder.add_edge("documentation_handler", END)
builder.add_edge("general_handler", END)

ticket_graph = builder.compile()

print("Ticket routing graph compiled")

In [ ]:
examples = [
    "Production deployment failed with error 503.",
    "Please update the Kubernetes onboarding guide.",
    "Can someone explain this configuration?"
]

for ticket in examples:
    result = ticket_graph.invoke({
        "ticket": ticket,
        "category": "",
        "response": ""
    })

    print("Ticket:", ticket)
    print("Category:", result["category"])
    print("Response:", result["response"])
    print("---")

## 6. Visualize Graph as Mermaid

LangGraph can produce a Mermaid diagram.

This is useful for documentation and explaining workflows.

In [ ]:
print(ticket_graph.get_graph().draw_mermaid())

## 7. LLM Node

A LangGraph node can call an LLM.

The node is still just a Python function:

```text
state → node → model call → state update
```

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=model_name,
    temperature=0
)

print("LLM initialized")

In [ ]:
class ExplanationState(TypedDict):
    topic: str
    audience: str
    explanation: str


def explain_node(state: ExplanationState):
    prompt = f"""
Explain {state["topic"]} to {state["audience"]}.
Use maximum 5 bullet points.
Keep it practical.
"""
    response = llm.invoke(prompt)
    return {"explanation": response.content}


builder = StateGraph(ExplanationState)

builder.add_node("explain", explain_node)
builder.add_edge(START, "explain")
builder.add_edge("explain", END)

explanation_graph = builder.compile()

print("Explanation graph compiled")

In [ ]:
result = explanation_graph.invoke({
    "topic": "LangGraph state",
    "audience": "software engineers",
    "explanation": ""
})

print(result["explanation"])

## 8. MessagesState

For chatbots and agents, LangGraph provides a convenient state called `MessagesState`.

It stores a list of messages and appends new messages automatically.

This connects directly to Session 4.

In [ ]:
from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage, AIMessage

def chatbot_node(state: MessagesState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


builder = StateGraph(MessagesState)

builder.add_node("chatbot", chatbot_node)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

chat_graph = builder.compile()

print("Chat graph compiled")

In [ ]:
result = chat_graph.invoke({
    "messages": [
        HumanMessage(content="Explain LangGraph in one short paragraph.")
    ]
})

print(result["messages"][-1].content)

## 9. Short-Term Memory with Checkpointer

A checkpointer saves graph state between calls.

Important concepts:

```text
checkpointer = stores graph state
thread_id = identifies one conversation/thread
```

Same `thread_id` continues the same memory.
Different `thread_id` starts a separate conversation.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

chat_graph_with_memory = builder.compile(checkpointer=checkpointer)

print("Chat graph with memory compiled")

In [ ]:
thread_config = {"configurable": {"thread_id": "session-5-demo"}}

result = chat_graph_with_memory.invoke(
    {
        "messages": [
            HumanMessage(content="My name is Vlad and I am learning LangGraph.")
        ]
    },
    config=thread_config
)

print(result["messages"][-1].content)

In [ ]:
result = chat_graph_with_memory.invoke(
    {
        "messages": [
            HumanMessage(content="What am I learning?")
        ]
    },
    config=thread_config
)

print(result["messages"][-1].content)

## 10. Thread Isolation

Now use a different `thread_id`.

The graph should not have the previous conversation memory.

In [ ]:
different_thread_config = {"configurable": {"thread_id": "different-session"}}

result = chat_graph_with_memory.invoke(
    {
        "messages": [
            HumanMessage(content="What am I learning?")
        ]
    },
    config=different_thread_config
)

print(result["messages"][-1].content)

## 11. Practical Workflow: Support Ticket Assistant

Now let's build a more practical workflow:

```text
START
  ↓
classify
  ↓
route
  ↓
technical_handler / docs_handler / escalation_handler
  ↓
draft_summary
  ↓
END
```

This is a simplified but realistic enterprise pattern.

In [ ]:
class SupportState(TypedDict):
    ticket: str
    category: str
    priority: str
    analysis: str
    summary: str


def classify_support_ticket(state: SupportState):
    ticket = state["ticket"].lower()

    if any(word in ticket for word in ["production", "down", "critical", "urgent"]):
        return {"category": "technical", "priority": "high"}

    if any(word in ticket for word in ["guide", "readme", "documentation", "how to"]):
        return {"category": "documentation", "priority": "medium"}

    return {"category": "general", "priority": "low"}


def route_support_ticket(state: SupportState) -> Literal["technical", "documentation", "general"]:
    return state["category"]


def technical_handler(state: SupportState):
    return {
        "analysis": (
            "Technical issue. Suggested steps: check monitoring, inspect recent deployments, "
            "collect logs, and escalate if production impact is confirmed."
        )
    }


def docs_handler(state: SupportState):
    return {
        "analysis": (
            "Documentation request. Suggested steps: search existing docs, identify missing section, "
            "and update the relevant README or runbook."
        )
    }


def general_support_handler(state: SupportState):
    return {
        "analysis": (
            "General request. Suggested steps: ask for additional context and route to the right owner."
        )
    }


def draft_summary_node(state: SupportState):
    summary = f"""
Ticket Summary:
- Category: {state["category"]}
- Priority: {state["priority"]}
- Analysis: {state["analysis"]}
"""
    return {"summary": summary.strip()}

In [ ]:
builder = StateGraph(SupportState)

builder.add_node("classify", classify_support_ticket)
builder.add_node("technical_handler", technical_handler)
builder.add_node("docs_handler", docs_handler)
builder.add_node("general_handler", general_support_handler)
builder.add_node("draft_summary", draft_summary_node)

builder.add_edge(START, "classify")

builder.add_conditional_edges(
    "classify",
    route_support_ticket,
    {
        "technical": "technical_handler",
        "documentation": "docs_handler",
        "general": "general_handler",
    }
)

builder.add_edge("technical_handler", "draft_summary")
builder.add_edge("docs_handler", "draft_summary")
builder.add_edge("general_handler", "draft_summary")
builder.add_edge("draft_summary", END)

support_graph = builder.compile()

print("Support workflow graph compiled")

In [ ]:
support_examples = [
    "Production API is down after the latest deployment.",
    "How to update the Kubernetes namespace onboarding guide?",
    "I have a question about the training repository."
]

for ticket in support_examples:
    result = support_graph.invoke({
        "ticket": ticket,
        "category": "",
        "priority": "",
        "analysis": "",
        "summary": ""
    })

    print(result["summary"])
    print("---")

## 12. Optional: Add LLM Drafting

The previous workflow used deterministic Python logic.

In real applications, you can add LLM nodes for:

- classification
- analysis
- response drafting
- summarization
- risk assessment

But keep deterministic code where deterministic code is better.

In [ ]:
class LLMSupportState(TypedDict):
    ticket: str
    category: str
    priority: str
    analysis: str
    final_response: str


def llm_draft_response_node(state: LLMSupportState):
    prompt = f"""
You are an internal engineering support assistant.

Ticket:
{state["ticket"]}

Category:
{state["category"]}

Priority:
{state["priority"]}

Analysis:
{state["analysis"]}

Draft a concise response with:
- short summary
- recommended next steps
- whether escalation is needed
"""
    response = llm.invoke(prompt)
    return {"final_response": response.content}

## 13. Human-in-the-Loop Concept

Some workflows should pause before risky actions.

Examples:

- send email
- approve deployment
- delete resource
- update production config

LangGraph supports interrupts for human-in-the-loop workflows.

We will keep this as a concept here and can build it in a later advanced session.

## 14. Exercises

### Exercise 1

Add a new category to the ticket graph:

```text
security
```

Route tickets containing:

```text
vulnerability, CVE, secret, token, password
```

to a new `security_handler`.

### Exercise 2

Add a new field to state:

```python
owner: str
```

Set owner based on category.

### Exercise 3

Add an LLM node that drafts the final response.

## 15. Discussion Questions

1. What is the difference between a node and an edge?
2. Why is state important?
3. When would you use conditional edges?
4. What does a checkpointer do?
5. Why is `thread_id` important?
6. When should we use deterministic code instead of an LLM node?
7. When would human approval be required?

## 16. Key Takeaways

- LangGraph models workflows as graphs.
- State is shared data.
- Nodes do work.
- Edges control flow.
- Conditional edges enable branching.
- `MessagesState` is useful for chatbots.
- Checkpointers provide short-term memory.
- LangGraph is useful when agents need state, control, and multiple steps.

## 17. Official Documentation

- LangGraph overview: https://docs.langchain.com/oss/python/langgraph/overview
- Graph API: https://docs.langchain.com/oss/python/langgraph/graph-api
- Persistence: https://docs.langchain.com/oss/python/langgraph/persistence
- Thinking in LangGraph: https://docs.langchain.com/oss/python/langgraph/thinking-in-langgraph
- Interrupts: https://docs.langchain.com/oss/python/langgraph/interrupts